In [ ]:
!pip install scikeras
!pip install --upgrade tensorflow keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 74.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.7/347.7 kB 28.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
  Attempting uninstall: keras
    Found existing installation: keras 2.15.0
    Uninstalling keras-2.15.0:
      Successfully uninstalled keras-2.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.4.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.3/601.3 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 76.1 MB/s eta 0:00:0

In [ ]:
#!pip install yfinance
# Thêm các thư viện cần thiết
import pandas as pd
from pandas_datareader import data as pdr
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from tensorflow.keras.models import Model
from keras.layers import LSTM, Dense, Bidirectional, Dropout, Input, Concatenate
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import time
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import History
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Activation
from keras.layers import Conv1D, MaxPooling1D, Flatten, AveragePooling1D
!pip install keras-self-attention
from keras_self_attention import SeqSelfAttention
from tensorflow.keras.layers import MultiHeadAttention
!pip install attention
from attention import Attention

  Preparing metadata (setup.py) ... done
  Created wheel for keras-self-attention: filename=keras_self_attention-0.51.0-py3-none-any.whl size=18894 sha256=d7d07d4fd9c11bad1057612a341b4a69e9360a79921d77b5953a42ad0ee7e94f
  Stored in directory: /root/.cache/pip/wheels/b8/f7/24/607b483144fb9c47b4ba2c5fba6b68e54aeee2d5bf6c05302e
Successfully built keras-self-attention


In [ ]:
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasRegressor

In [ ]:
data_ORCL = pd.read_csv('ORCL.csv')

data_ORCL

,Date,Close,Open,High,Low,Adj Close,Volume
0,2019-12-31,52.980000,52.570000,53.000000,52.549999,49.155796,7094500
1,2020-01-02,53.950001,53.270000,53.959999,53.230000,50.055779,13899600
2,2020-01-03,53.759998,52.990002,54.049999,52.950001,49.879490,11026700
3,2020-01-06,54.040001,53.360001,54.200001,53.349998,50.139278,10982400
4,2020-01-07,54.160000,53.889999,54.330002,53.610001,50.250622,12015400
...,...,...,...,...,...,...,...
1105,2024-05-22,124.599998,124.629997,125.160004,123.300003,124.249191,5705000
1106,2024-05-23,124.089996,126.550003,126.699997,123.160004,123.740623,6108600
1107,2024-05-24,122.910004,123.419998,123.510002,121.419998,122.563950,7166100
1108,2024-05-28,124.489998,123.239998,124.820000,123.010002,124.139496,6911400


In [ ]:
data = data_ORCL
epoch = 500
#columns = ['Close']
columns = ['Close', 'Open', 'High', 'Low', 'Volume']
#columns = ['Close', 'MACD', 'MFI', 'RSI', 'ATR']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR', 'USD/VND', 'IR', 'CFI', 'Broad money M2']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR', 'USD/VND', 'IR', 'CFI']

data = data[columns]

train_size = int(len(data) * 0.9)

train = data[:train_size]
test = data['Close']
test = data[train_size:]

train = train[columns]

scaler = MinMaxScaler(feature_range = (0, 1))

# train_scaled = scaler.fit_transform(train)
# test_scaled = scaler.transform(data)
scaled = scaler.fit_transform(data)
train_scaled = scaled[:int(train_size)]

test_data_lb_7day = scaled[int(train_size) - 7: , :]
test_data_lb_14day = scaled[int(train_size) - 14: , :]
test_data_lb_21day = scaled[int(train_size) - 21: , :]


lb = 7
X_train = []
Y_train = []

for i in range(lb, len(train_scaled)):
    X_train.append(train_scaled[i - lb:i, :])
    Y_train.append(train_scaled[i, 0])

X_train, Y_train = np.array(X_train), np.array(Y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], len(columns)))
x_train, x_val, y_train, y_val = train_test_split(X_train, Y_train, test_size=0.1)

units = [150, 200, 384]
batch_size = [32, 64, 96, 128]
dropout_rate = [0.1, 0.2, 0.3, 0.4]
num_layers = [1, 2, 3]


def create_parallel_lstm_model(units, dropout_rate, num_layers):
    input_layer = Input(shape=(x_train.shape[1], x_train.shape[2]))

    cnn_1 = input_layer
    # First branch of LSTM layers
    cnn_1 = Conv1D(filters=64, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=128, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    #cnn_1 = Conv1D(filters=128, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=64, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = AveragePooling1D(pool_size=2)(cnn_1)
    for i in range(num_layers):
        if i == num_layers - 1:
            lstm_1 = Bidirectional(LSTM(units=units, return_sequences=True))(cnn_1)
        else:
            lstm_1 = Bidirectional(LSTM(units=units, return_sequences=True))(cnn_1)
            lstm_1 = Dropout(dropout_rate)(lstm_1)
    lstm_1 = SeqSelfAttention(attention_activation='elu')(lstm_1)

    # Concatenate the outputs of both branches
    combined = Concatenate()([lstm_1, lstm_1])

    combined = Flatten()(combined)

    # Final dense layer
    combined = Dense(units=128, activation='sigmoid')(combined)
    output = Dense(units=1)(combined)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(optimizer='adam', loss='mse')

    return model

# define the grid search parameters
param_grid = dict(units = units, dropout_rate =  dropout_rate, batch_size=batch_size, num_layers = num_layers)

model = KerasRegressor(build_fn=create_parallel_lstm_model, units = units, dropout_rate =  dropout_rate, num_layers = num_layers)

grid = GridSearchCV(estimator=model, param_grid=param_grid,cv=2,refit=False, scoring="neg_mean_squared_error", n_jobs = -1)

grid_result = grid.fit(x_train, y_train)

# summarize results
print("Best accuracy of: %f using %s" % (grid_result.best_score_,
                                         grid_result.best_params_))

results = []
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    results.append((mean, stdev, param))
results.sort(key=lambda x: x[0], reverse=True)
for mean, stdev, param in results:
    print("%f (%f) with: %r" % (mean, stdev, param))


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


In [ ]:
data = data_ORCL
epoch = 500
#columns = ['Close']
columns = ['Close', 'Open', 'High', 'Low', 'Volume']
#columns = ['Close', 'MACD', 'MFI', 'RSI', 'ATR']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR', 'USD/VND', 'IR', 'CFI', 'Broad money M2']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR', 'USD/VND', 'IR', 'CFI']

data = data[columns]

train_size = int(len(data) * 0.9)

train = data[:train_size]
test = data['Close']
test = data[train_size:]

train = train[columns]

scaler = MinMaxScaler(feature_range = (0, 1))

# train_scaled = scaler.fit_transform(train)
# test_scaled = scaler.transform(data)
scaled = scaler.fit_transform(data)
train_scaled = scaled[:int(train_size)]

test_data_lb_7day = scaled[int(train_size) - 7: , :]
test_data_lb_14day = scaled[int(train_size) - 14: , :]
test_data_lb_21day = scaled[int(train_size) - 21: , :]


lb = 7
X_train = []
Y_train = []

for i in range(lb, len(train_scaled)):
    X_train.append(train_scaled[i - lb:i, :])
    Y_train.append(train_scaled[i, 0])

X_train, Y_train = np.array(X_train), np.array(Y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], len(columns)))
x_train, x_val, y_train, y_val = train_test_split(X_train, Y_train, test_size=0.1)

units = [128, 150, 200, 256, 384]
batch_size = [32, 64, 96, 128, 160]
dropout_rate = [0.1, 0.2, 0.3, 0.4]
num_layers = [1, 2, 3]


def create_parallel_lstm_model(units, dropout_rate, num_layers):
    input_layer = Input(shape=(x_train.shape[1], x_train.shape[2]))

    cnn_1 = input_layer
    # First branch of LSTM layers
    cnn_1 = Conv1D(filters=64, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=128, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    #cnn_1 = Conv1D(filters=128, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=64, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = AveragePooling1D(pool_size=2)(cnn_1)
    for i in range(num_layers):
        if i == num_layers - 1:
            lstm_1 = Bidirectional(LSTM(units=units, return_sequences=True))(cnn_1)
        else:
            lstm_1 = Bidirectional(LSTM(units=units, return_sequences=True))(cnn_1)
            lstm_1 = Dropout(dropout_rate)(lstm_1)
    lstm_1 = SeqSelfAttention(attention_activation='elu')(lstm_1)

    # Concatenate the outputs of both branches
    combined = Concatenate()([lstm_1, lstm_1])

    combined = Flatten()(combined)

    # Final dense layer
    combined = Dense(units=128, activation='sigmoid')(combined)
    output = Dense(units=1)(combined)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(optimizer='adam', loss='mse')

    return model

# define the grid search parameters
param_grid = dict(units = units, dropout_rate =  dropout_rate, batch_size=batch_size, num_layers = num_layers)

model = KerasRegressor(build_fn=create_parallel_lstm_model, units = units, dropout_rate =  dropout_rate, num_layers = num_layers)

grid = GridSearchCV(estimator=model, param_grid=param_grid,cv=2,refit=False, scoring="neg_mean_squared_error", n_jobs = -1)

grid_result = grid.fit(x_train, y_train)

# summarize results
print("Best accuracy of: %f using %s" % (grid_result.best_score_,
                                         grid_result.best_params_))

results = []
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    results.append((mean, stdev, param))
results.sort(key=lambda x: x[0], reverse=True)
for mean, stdev, param in results:
    print("%f (%f) with: %r" % (mean, stdev, param))


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()
/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best accuracy of: -0.003385 using {'batch_size': 32, 'dropout_rate': 0.3, 'num_layers': 4, 'units': 256}
-0.003385 (0.000229) with: {'batch_size': 32, 'dropout_rate': 0.3, 'num_layers': 4, 'units': 256}
-0.004526 (0.000172) with: {'batch_size': 96, 'dropout_rate': 0.2, 'num_layers': 2, 'units': 128}
-0.005181 (0.000207) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 1, 'units': 256}
-0.005853 (0.001952) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 3, 'units': 256}
-0.006570 (0.004085) with: {'batch_size': 32, 'dropout_rate': 0.1, 'num_layers': 1, 'units': 256}
-0.007648 (0.003155) with: {'batch_size': 32, 'dropout_rate': 0.1, 'num_layers': 2, 'units': 256}
-0.007838 (0.003670) with: {'batch_size': 128, 'dropout_rate': 0.3, 'num_layers': 2, 'units': 256}
-0.008150 (0.004194) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 4, 'units': 128}
-0.008764 (0.002614) with: {'batch_size': 32, 'dropout_rate': 0.3, 'num_layers': 1, 'units': 256}
-0.009313 (0

/usr/local/lib/python3.10/dist-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


In [ ]:
data = data_ORCL
epoch = 500
#columns = ['Close']
columns = ['Close', 'Open', 'High', 'Low', 'Volume']
#columns = ['Close', 'MACD', 'MFI', 'RSI', 'ATR']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR', 'USD/VND', 'IR', 'CFI', 'Broad money M2']
#columns = ['Close', 'Open', 'High', 'Low', 'Volume', 'MACD', 'MFI', 'RSI', 'ATR', 'USD/VND', 'IR', 'CFI']

data = data[columns]

train_size = int(len(data) * 0.9)

train = data[:train_size]
test = data['Close']
test = data[train_size:]

train = train[columns]

scaler = MinMaxScaler(feature_range = (0, 1))

# train_scaled = scaler.fit_transform(train)
# test_scaled = scaler.transform(data)
scaled = scaler.fit_transform(data)
train_scaled = scaled[:int(train_size)]

test_data_lb_7day = scaled[int(train_size) - 7: , :]
test_data_lb_14day = scaled[int(train_size) - 14: , :]
test_data_lb_21day = scaled[int(train_size) - 21: , :]


lb = 7
X_train = []
Y_train = []

for i in range(lb, len(train_scaled)):
    X_train.append(train_scaled[i - lb:i, :])
    Y_train.append(train_scaled[i, 0])

X_train, Y_train = np.array(X_train), np.array(Y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], len(columns)))
x_train, x_val, y_train, y_val = train_test_split(X_train, Y_train, test_size=0.1)

units = [128, 150, 200, 256, 384]
batch_size = [32, 64, 96, 128, 160]
dropout_rate = [0.1, 0.2, 0.3, 0.4]
num_layers = [1, 2, 3]


def create_parallel_lstm_model(units, dropout_rate, num_layers):
    input_layer = Input(shape=(x_train.shape[1], x_train.shape[2]))

    cnn_1 = input_layer
    # First branch of LSTM layers
    cnn_1 = Conv1D(filters=64, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=128, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=128, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = Conv1D(filters=64, kernel_size=3, padding = 'same', activation='relu')(cnn_1)
    cnn_1 = AveragePooling1D(pool_size=2)(cnn_1)
    for i in range(num_layers):
        if i == num_layers - 1:
            lstm_1 = Bidirectional(LSTM(units=units, return_sequences=True))(cnn_1)
        else:
            lstm_1 = Bidirectional(LSTM(units=units, return_sequences=True))(cnn_1)
            lstm_1 = Dropout(dropout_rate)(lstm_1)
    lstm_1 = SeqSelfAttention(attention_activation='elu')(lstm_1)

    # Concatenate the outputs of both branches
    combined = Concatenate()([lstm_1, lstm_1])

    combined = Flatten()(combined)

    # Final dense layer
    combined = Dense(units=128, activation='sigmoid')(combined)
    output = Dense(units=1)(combined)

    model = Model(inputs=input_layer, outputs=output)

    model.compile(optimizer='adam', loss='mse')

    return model

# define the grid search parameters
param_grid = dict(units = units, dropout_rate =  dropout_rate, batch_size=batch_size, num_layers = num_layers)

model = KerasRegressor(build_fn=create_parallel_lstm_model, units = units, dropout_rate =  dropout_rate, num_layers = num_layers)

grid = GridSearchCV(estimator=model, param_grid=param_grid,cv=2,refit=False, scoring="neg_mean_squared_error", n_jobs = -1)

grid_result = grid.fit(x_train, y_train)

# summarize results
print("Best accuracy of: %f using %s" % (grid_result.best_score_,
                                         grid_result.best_params_))

results = []
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    results.append((mean, stdev, param))
results.sort(key=lambda x: x[0], reverse=True)
for mean, stdev, param in results:
    print("%f (%f) with: %r" % (mean, stdev, param))


/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/joblib/externals/loky/backend/fork_exec.py:38: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid = os.fork()


Best accuracy of: -0.003048 using {'batch_size': 32, 'dropout_rate': 0.1, 'num_layers': 2, 'units': 128}
-0.003048 (0.000753) with: {'batch_size': 32, 'dropout_rate': 0.1, 'num_layers': 2, 'units': 128}
-0.004894 (0.003072) with: {'batch_size': 32, 'dropout_rate': 0.3, 'num_layers': 1, 'units': 256}
-0.006153 (0.003242) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 1, 'units': 128}
-0.007434 (0.002053) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 4, 'units': 128}
-0.008071 (0.004905) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 1, 'units': 256}
-0.008532 (0.002611) with: {'batch_size': 32, 'dropout_rate': 0.1, 'num_layers': 3, 'units': 256}
-0.008536 (0.005273) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 2, 'units': 256}
-0.009062 (0.004942) with: {'batch_size': 32, 'dropout_rate': 0.2, 'num_layers': 3, 'units': 128}
-0.010771 (0.000471) with: {'batch_size': 32, 'dropout_rate': 0.1, 'num_layers': 3, 'units': 128}
-0.011391 (0.